# Harmonized EEG dataloader: one full epoch benchmark

This notebook demonstrates how the training pipeline consumes the Parquet manifest and Arrow signal shards generated by the harmonization step. It:

1. inspects the selected manifest view;
2. constructs the same `EEGDataModule` used by training;
3. reads and validates one batch;
4. measures one **complete dataloader epoch**;
5. optionally compares worker counts.

The timing excludes model forward/backward computation. It includes dataloader iterator startup, Arrow reads, decompression, tensor reconstruction, collation, and optional host-to-device transfer.


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import torch

from cbramod_experiments.data_harmonization import EEGDataModule
from cbramod_experiments.utils.data_benchmark import benchmark_dataloader_epoch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

DATA_PATH = Path(
    os.getenv(
        "DATALOADER_DATA",
        PROJECT_ROOT / "outputs/data/harmonized/hbn/manifest.parquet",
    )
)
BACKEND = os.getenv("DATALOADER_BACKEND", "arrow_streaming")
SPLIT = None
REQUIRE_LABELS = False
BATCH_SIZE = int(os.getenv("DATALOADER_BATCH_SIZE", "128"))
NUM_WORKERS = int(os.getenv("DATALOADER_NUM_WORKERS", "4"))
PREFETCH_FACTOR = int(os.getenv("DATALOADER_PREFETCH_FACTOR", "2"))
SHUFFLE_BUFFER = int(os.getenv("DATALOADER_SHUFFLE_BUFFER", "1024"))
DEVICE = os.getenv("DATALOADER_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
EPOCH = int(os.getenv("DATALOADER_EPOCH", "0"))

print(
    {
        "data": str(DATA_PATH),
        "backend": BACKEND,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "device": DEVICE,
    }
)
assert DATA_PATH.exists(), f"Missing data path: {DATA_PATH}"

{'data': '/home/yassir/projects/cbramod_experiments/outputs/data/harmonized/hbn/manifest.parquet', 'backend': 'arrow_streaming', 'batch_size': 128, 'num_workers': 4, 'device': 'cuda'}


## 1. Inspect the harmonized manifest

The manifest contains metadata and physical locations; dense tensors remain in Arrow shards. Predicate filtering selects a split without scanning signal payloads.


In [2]:
if DATA_PATH.suffix == ".parquet":
    manifest = pq.read_table(DATA_PATH)
    manifest_df = manifest.to_pandas()
    display(manifest_df.head())
    display(
        pd.DataFrame(
            {
                "rows": [len(manifest_df)],
                "unique_samples": [manifest_df["sample_id"].nunique()],
                "shards": [manifest_df["shard_path"].nunique()],
                "datasets": [sorted(manifest_df["dataset_id"].unique().tolist())],
                "splits": [manifest_df["split"].value_counts(dropna=False).to_dict()],
            }
        )
    )
else:
    print("HDF5 selected: manifest inspection is skipped.")

,sample_id,dataset_id,subject_id,session_id,task,split,shard_path,record_batch,row_in_batch,sampling_rate_hz,...,channel_mask,label,start_seconds,duration_seconds,source_uri,source_format,preprocessing_version,amplitude_scale,quality_flags,metadata_json
0,hbn:sub-NDARAC904DMU_task-DespicableMe:sample-...,hbn,sub-NDARAC904DMU,NaN,task-DespicableMe,NaN,shards/shard-000000.arrow,0,0,500.0,...,"[True, True, True, True, True, True, True, Tru...",NaN,0.0,4.0,resources/data/hbn/sub-NDARAC904DMU/eeg/sub-ND...,set,harmonized-eeg-v1,100.0,"[flat_channel, extreme_amplitude]","{""channels"": [{""name"": ""E1"", ""type"": ""EEG"", ""u..."
1,hbn:sub-NDARAC904DMU_task-DespicableMe:sample-...,hbn,sub-NDARAC904DMU,NaN,task-DespicableMe,NaN,shards/shard-000000.arrow,0,1,500.0,...,"[True, True, True, True, True, True, True, Tru...",NaN,4.0,4.0,resources/data/hbn/sub-NDARAC904DMU/eeg/sub-ND...,set,harmonized-eeg-v1,100.0,"[flat_channel, extreme_amplitude]","{""channels"": [{""name"": ""E1"", ""type"": ""EEG"", ""u..."
2,hbn:sub-NDARAC904DMU_task-DespicableMe:sample-...,hbn,sub-NDARAC904DMU,NaN,task-DespicableMe,NaN,shards/shard-000000.arrow,0,2,500.0,...,"[True, True, True, True, True, True, True, Tru...",NaN,8.0,4.0,resources/data/hbn/sub-NDARAC904DMU/eeg/sub-ND...,set,harmonized-eeg-v1,100.0,"[flat_channel, extreme_amplitude]","{""channels"": [{""name"": ""E1"", ""type"": ""EEG"", ""u..."
3,hbn:sub-NDARAC904DMU_task-DespicableMe:sample-...,hbn,sub-NDARAC904DMU,NaN,task-DespicableMe,NaN,shards/shard-000000.arrow,0,3,500.0,...,"[True, True, True, True, True, True, True, Tru...",NaN,12.0,4.0,resources/data/hbn/sub-NDARAC904DMU/eeg/sub-ND...,set,harmonized-eeg-v1,100.0,"[flat_channel, extreme_amplitude]","{""channels"": [{""name"": ""E1"", ""type"": ""EEG"", ""u..."
4,hbn:sub-NDARAC904DMU_task-DespicableMe:sample-...,hbn,sub-NDARAC904DMU,NaN,task-DespicableMe,NaN,shards/shard-000000.arrow,0,4,500.0,...,"[True, True, True, True, True, True, True, Tru...",NaN,16.0,4.0,resources/data/hbn/sub-NDARAC904DMU/eeg/sub-ND...,set,harmonized-eeg-v1,100.0,"[flat_channel, extreme_amplitude]","{""channels"": [{""name"": ""E1"", ""type"": ""EEG"", ""u..."


,rows,unique_samples,shards,datasets,splits
0,105151,105151,32,[hbn],{nan: 105151}


## 2. Construct the training dataloader and read one batch

`arrow_streaming` partitions shards across dataloader workers and performs bounded-buffer shuffling. `arrow` offers random access with record-batch-aware sampling. Both return the same `(signals, labels)` contract.


In [3]:
module = EEGDataModule(
    DATA_PATH,
    backend=BACKEND,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.startswith("cuda"),
    persistent_workers=NUM_WORKERS > 0,
    seed=0,
    streaming_shuffle_buffer_size=SHUFFLE_BUFFER,
    prefetch_factor=PREFETCH_FACTOR,
    require_labels=REQUIRE_LABELS,
)

loader = module.loader(
    SPLIT,
    shuffle=False,  # better for a deterministic throughput benchmark
)

if hasattr(loader.dataset, "set_epoch"):
    loader.dataset.set_epoch(EPOCH)

signals, labels = next(iter(loader))
print("signals:", tuple(signals.shape), signals.dtype)
print("labels: ", tuple(labels.shape), labels.dtype)
print("signal range:", float(signals.min()), float(signals.max()))
print("first labels:", labels[:10].tolist())

signals: (128, 129, 2000) torch.float32
labels:  (128,) torch.int64
signal range: -1413.6104736328125 1938.9378662109375
first labels: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1]


## 2. Ensure all shards are balanced

Uneven shards can lead to performance degradation over time.

In [4]:
from pathlib import Path
import random

import pyarrow.parquet as pq


MANIFEST = Path("outputs/data/harmonized/hbn/manifest.parquet")
NUM_WORKERS = 4
SEED = 0

df = pq.read_table(
    MANIFEST,
    columns=[
        "shard_path",
        "num_channels",
        "num_samples",
    ],
).to_pandas()

df["signal_bytes"] = (
    df["num_channels"] * df["num_samples"] * 4  # float32
)

weights = df.groupby("shard_path").agg(
    examples=("shard_path", "size"),
    signal_bytes=("signal_bytes", "sum"),
)

for epoch in range(15):
    shards = list(weights.index)
    random.Random(SEED + epoch).shuffle(shards)

    worker_bytes = [0] * NUM_WORKERS
    worker_examples = [0] * NUM_WORKERS

    for index, shard in enumerate(shards):
        worker = index % NUM_WORKERS
        worker_bytes[worker] += int(weights.loc[shard, "signal_bytes"])
        worker_examples[worker] += int(weights.loc[shard, "examples"])

    imbalance = max(worker_bytes) / max(1, min(worker_bytes))

    print(
        f"epoch={epoch:02d}",
        f"examples={worker_examples}",
        "GiB=",
        [round(value / 1024**3, 2) for value in worker_bytes],
        f"imbalance={imbalance:.2f}x",
    )

epoch=00 examples=[26306, 22927, 28238, 27680] GiB= [25.28, 22.04, 27.14, 26.6] imbalance=1.23x
epoch=01 examples=[21216, 21708, 29459, 32768] GiB= [20.39, 20.86, 28.31, 31.49] imbalance=1.54x
epoch=02 examples=[32648, 20756, 27124, 24623] GiB= [31.38, 19.95, 26.07, 23.67] imbalance=1.57x
epoch=03 examples=[20363, 25874, 29855, 29059] GiB= [19.57, 24.87, 28.69, 27.93] imbalance=1.47x
epoch=04 examples=[24612, 27448, 22612, 30479] GiB= [23.66, 26.38, 21.73, 29.29] imbalance=1.35x
epoch=05 examples=[24831, 25933, 28513, 25874] GiB= [23.87, 24.92, 27.4, 24.87] imbalance=1.15x
epoch=06 examples=[32768, 23475, 21894, 27014] GiB= [31.49, 22.56, 21.04, 25.96] imbalance=1.50x
epoch=07 examples=[24547, 29482, 28349, 22773] GiB= [23.59, 28.34, 27.25, 21.89] imbalance=1.29x
epoch=08 examples=[32173, 21679, 28954, 22345] GiB= [30.92, 20.84, 27.83, 21.48] imbalance=1.48x
epoch=09 examples=[27908, 25826, 22355, 29062] GiB= [26.82, 24.82, 21.49, 27.93] imbalance=1.30x
epoch=10 examples=[21756, 29531,

## 3. Measure one complete epoch

The benchmark verifies that the number of observed examples equals the selected dataset size. `first_batch_seconds` includes worker startup and the first I/O operation. `epoch_seconds` includes the complete iterator lifetime. Throughput counts uncompressed signal tensor bytes, not total filesystem traffic.


In [5]:
result = benchmark_dataloader_epoch(
    DATA_PATH,
    backend=BACKEND,
    split=SPLIT,
    output_path=PROJECT_ROOT / "outputs/benchmarks/dataloader_epoch.json",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    streaming_shuffle_buffer_size=SHUFFLE_BUFFER,
    epoch=EPOCH,
    device=DEVICE,
    show_progress=True,
    require_labels=REQUIRE_LABELS,
)

result_df = pd.DataFrame([result.to_dict()]).T.rename(columns={0: "value"})
display(result_df)
assert result.complete_epoch, (
    f"Incomplete epoch: observed {result.observed_examples}, "
    f"expected {result.expected_examples}"
)

arrow_streaming:None epoch 0:   0%|          | 0/105151 [00:00<?, ?example/s]

,value
data_path,/home/yassir/projects/cbramod_experiments/outp...
backend,arrow_streaming
split,None
device,cuda
epoch,0
batch_size,128
num_workers,4
prefetch_factor,2
pin_memory,True
persistent_workers,True


## 4. Optional worker-count sweep

Enable this only when you want to benchmark several complete epochs. More workers are not always faster: storage contention, decompression, Python multiprocessing, and CPU thread oversubscription can reduce throughput.


In [ ]:
RUN_WORKER_SWEEP = False
WORKER_COUNTS = [0, 2, 4, 8]

sweep_rows = []
if RUN_WORKER_SWEEP:
    for workers in WORKER_COUNTS:
        benchmark = benchmark_dataloader_epoch(
            DATA_PATH,
            backend=BACKEND,
            split=SPLIT,
            batch_size=BATCH_SIZE,
            num_workers=workers,
            prefetch_factor=PREFETCH_FACTOR,
            streaming_shuffle_buffer_size=SHUFFLE_BUFFER,
            epoch=EPOCH,
            device=DEVICE,
            show_progress=True,
            require_labels=REQUIRE_LABELS,
        )
        sweep_rows.append(benchmark.to_dict())

    sweep_df = pd.DataFrame(sweep_rows)
    display(
        sweep_df[
            [
                "num_workers",
                "epoch_seconds",
                "first_batch_seconds",
                "examples_per_second",
                "mebibytes_per_second",
            ]
        ]
    )

    ax = sweep_df.plot(
        x="num_workers",
        y="examples_per_second",
        marker="o",
        title="Full-epoch dataloader throughput",
    )
    ax.set_ylabel("examples / second")
    plt.show()
else:
    print("Set RUN_WORKER_SWEEP=True to run multiple full epochs.")

arrow_streaming:None epoch 0:   0%|          | 0/105151 [00:00<?, ?example/s]

KeyboardInterrupt: 

## Interpreting the result

- **Loader build time** measures manifest filtering and dataset/loader construction.
- **Iterator startup time** measures iterator and worker initialization before requesting the first batch.
- **First-batch time** includes startup plus the first read/decode/collation operation.
- **Epoch time** measures the complete data-only epoch.
- **Examples/s and MiB/s** are end-to-end application-level rates.

This is a data-pipeline benchmark, not a training benchmark. A separate model benchmark is needed to determine whether data loading overlaps sufficiently with GPU computation. On a cluster, run one benchmark per rank and aggregate throughput; only rank 0 should display progress.
